<a href="https://colab.research.google.com/github/peterlpk10-lang/atividades_individuais_poo/blob/main/Lab_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SQLAlchemy ORM (Mapeamento Objeto-Relacional)

```
     +-----------------+                  +-------------------+
     |                 |                  |                   |
     |   Aplicação     |  <--- ORM --->   |   Banco de Dados  |
     |                 |                  |                   |
     +-----------------+                  +-------------------+
            |                                       |
            |    +-----------------------------+    |
            +--->|  Classe    <-> Tabela       |<---+
                 |  Instância <-> Registro     |
                 |  Métodos   <-> SQL Queries  |
                 +-----------------------------+
```


Estruturalmente a Orientação a Objetos é muito diferente do armazenamento de dado em bancos relacionais, com direito até a uma [página no Wikipédia sobre essa incompatibilidade](https://en.wikipedia.org/wiki/Object%E2%80%93relational_impedance_mismatch). Com o propósito de reduzir esse distanciamento, cria-se o conceito de ORM, um módulo responsável por abstrair os dados e as operações de bancos de dados, criando uma interface de comunicação entre a linguagem de programação e o banco. Nessa estrutura temos:
* Classes representando tabelas e suas relações
* Instância de classe representado um registro (uma linha da tabela)
* Métodos abstraindo queries SQL.

Uma das principais vantagens dessa estrutura é a abstração da linguagem de banco, o que aumenta o encapsulamento. Quem desenvolve não precisa nem saber (nem se preocupar com) a linguagem específica SQL utilizada, já que o ORM permite se conectar com diferentes bancos, mantendo as sintaxes de modelagem de dados e queries SQL.

O código a seguir é um exemplo de modelagem de tabelas usando o paradigma orientado a objetos.



In [ ]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy import select, update, delete, insert, func, asc, desc
from sqlalchemy.orm import declarative_base, sessionmaker

#### criando a conexão
engine = create_engine('sqlite:///livraria.db')

#### criando uma tabela
Base = declarative_base()
class Livro(Base):
    __tablename__ = 'livros'
    id = Column(Integer, primary_key=True)
    titulo = Column(String)
    autor = Column(String)
    ano = Column(Integer)

    # Define como o objeto aparece no print()
    def __repr__(self):
        return f"Livro({self.id}, {self.titulo!r}, {self.autor!r}, {self.ano})"

# Apaga e recria a tabela: assim podemos reexecutar o notebook sem duplicar registros
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

### Session

Para interagir com o banco de dados iniciamos um objeto `Session`, que será nosso intermediário para inserções, consultas, etc.

Funções que alteram o estado do banco e ficam pendentes na sessão para serem concretizadas (ou não) após um commit:
* `add(<obj>)`: Adiciona um objeto à sessão para ser persistido no banco de dados.
* `add_all([<obj>])`: Adiciona a coleção de objetos à sessão para ser persistido no banco de dados.
* `delete(<obj>)`: Marca uma instância como deletada.

Funções que alteram o estado da transação:
* `commit()`: Libera as alterações pendentes e confirma a transação atual. Após o commit, os atributos dos objetos são marcados como expirados e recarregados do banco no próximo acesso.
* `rollback()`: Reverte a transação atual em andamento.

Para encerrar a sessão:
* `close()`: Encerra a sessão. É invocado automaticamente quando a sessão é aberta em uma estrutura de contexto `with`.

#### SQL x SQLAlchemy
| Operação | SQL | SQLAlchemy |
|---|---|---|
| Inserir | `INSERT INTO livros (titulo, autor, ano) VALUES ('1984', 'George Orwell', 1949);` | `session.add(Livro(titulo='1984', autor='George Orwell', ano=1949))`|
| Atualizar | `UPDATE livros SET ano = 1950 WHERE id = 2;` | `session.execute(update(Livro).where(Livro.id == 2).values(ano=1950))` |
| Deletar | `DELETE FROM livros WHERE id = 2;` | `session.execute(delete(Livro).where(Livro.id == 2))`|





No exemplo a seguir, vamos inserir 5 livros de uma só vez com `add_all()`:

In [ ]:
Session = sessionmaker(bind=engine) ### cursor // sessão
session = Session()

# adicionar uma linha no banco
session.add(Livro(titulo='Flores para Algernon', autor='Daniel Keyes', ano=1966)) # Insert into

# Cada objeto Livro é um registro (linha) da tabela
livros = [
    Livro(titulo='1984', autor='George Orwell', ano=1949),
    Livro(titulo='Dom Casmurro', autor='Machado de Assis', ano=1899),
    Livro(titulo='A Revolução dos Bichos', autor='George Orwell', ano=1945),
    Livro(titulo='Memórias Póstumas de Brás Cubas', autor='Machado de Assis', ano=1881),
]

session.add_all(livros) # Insert into
session.commit()
print(f"{len(livros)} livros inseridos")

5 livros inseridos


Agora vamos deletar apenas um deles (*Dom Casmurro*). Ficaremos com 4 livros para usar nas consultas:

In [ ]:
session.delete(livros[2])
session.commit()
print(f"Livro removido: {livros[2].titulo}")

Livro removido: Dom Casmurro


### Queries
Documentação: https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.Select<br>
Tutorial: https://docs.sqlalchemy.org/en/20/orm/queryguide/index.html

A tabela a seguir lista todos as operações sobre bancos de dados que aprenderemos nessa aula, mapeando do SQL que você já conhece para o SQLAlchemy.

| Operação | SQL | SQLAlchemy |
|---|---|---|
| Todos os registros | `SELECT * FROM livros;` | `select(Livro)` |
| Colunas específicas | `SELECT titulo, autor FROM livros;` | `select(Livro.titulo, Livro.autor)` |
| Filtro | `SELECT * FROM livros WHERE autor = 'George Orwell';` | `select(Livro).where(Livro.autor == 'George Orwell')` |
| Filtro com AND | `... WHERE autor = 'George Orwell' AND ano > 1946;` | `.where(Livro.autor == 'George Orwell', Livro.ano > 1946)` |
| Filtro com OR | `... WHERE autor = 'A' OR autor = 'B';` | `.where(or_(Livro.autor == 'A', Livro.autor == 'B'))` |
| Ordenação | `SELECT * FROM livros ORDER BY ano DESC;` | `select(Livro).order_by(desc(Livro.ano))` |
| Limite | `SELECT * FROM livros LIMIT 2;` | `select(Livro).limit(2)` |
| Contagem | `SELECT COUNT(id) FROM livros;` | `select(func.count(Livro.id))` |
| Agrupamento | `SELECT autor, COUNT(id) AS qtd FROM livros GROUP BY autor;` | `select(Livro.autor, func.count(Livro.id).label("qtd")).group_by(Livro.autor)` |


> Os comandos que **leem** dados (`select`) são executados com `session.execute()` ou `session.scalars()`. Os que **alteram** dados só valem de verdade após o `session.commit()`.

#### `session.execute()`
Para executar as instruções SQL que conheceremos a partir de agora podemos usar o método de sessão `execute()`, que recebe um objeto `statement` do SQLAlchemy. Ao capturar o resultado da execução, através de métodos como `.all()` ou `.first()`, retorna uma coleção de objetos `Row` (linha, em inglês, se referindo a uma linha da tabela).

#### SELECT
`select(Livro)` equivale a `SELECT * FROM livros`. Cada `Row` guarda o objeto na chave com o nome da classe (`linha.Livro`):

In [ ]:
query = select(Livro) # SELECT * FROM livros
res = session.execute(query).all()

for linha in res: # objeto Row
    print(linha.Livro.titulo)

1 Flores para Algernon
2 1984
4 A Revolução dos Bichos
5 Memórias Póstumas de Brás Cubas


Para selecionar apenas alguns atributos (colunas), passamos eles ao `select`. Agora cada `Row` tem os atributos pedidos:

In [ ]:
query = select(Livro.titulo, Livro.autor)

## cursor -> execute / fetchall
res = session.execute(query) # execute
res = res.all() # fetchall()

for linha in res:
    print(linha.titulo, '-', linha.autor)

Flores para Algernon - Daniel Keyes
1984 - George Orwell
A Revolução dos Bichos - George Orwell
Memórias Póstumas de Brás Cubas - Machado de Assis


Podemos incrementar nosso SELECT através de seus métodos. O primeiro é o `where()`, que define as condições da cláusula `WHERE`.

Dica: ao dar `print()` em uma query, vemos o SQL que o SQLAlchemy vai gerar.

In [ ]:
query = select(Livro).where(Livro.autor == 'George Orwell')

print(query)   # SQL gerado
print()
for linha in session.execute(query):
    print(linha.Livro)

SELECT livros.id, livros.titulo, livros.autor, livros.ano 
FROM livros 
WHERE livros.autor = :autor_1

Livro(2, '1984', 'George Orwell', 1949)
Livro(4, 'A Revolução dos Bichos', 'George Orwell', 1945)


Várias condições passadas ao `where()` são combinadas com `AND`. Para `OR`, use `or_()` (importado de `sqlalchemy`).

In [ ]:
from sqlalchemy import or_

# # AND: Orwell E publicado depois de 1946
query = select(Livro).where(Livro.autor == 'George Orwell', Livro.ano > 1946)
print("AND:", session.execute(query).all())

# OR: Daniel Keyes OU Machado de Assis
query = select(Livro).where(or_(Livro.autor == 'Daniel Keyes', Livro.autor == 'Machado de Assis'))
print("OR: ", session.execute(query).all())

AND: [(Livro(2, '1984', 'George Orwell', 1949),)]
OR:  [(Livro(1, 'Flores para Algernon', 'Daniel Keyes', 1966),), (Livro(5, 'Memórias Póstumas de Brás Cubas', 'Machado de Assis', 1881),)]


`order_by()` define a cláusula `ORDER BY`. Por padrão a ordem é crescente; para decrescente usamos `desc()`:

In [ ]:
# Do mais novo para o mais antigo
query = select(Livro.titulo, Livro.ano).order_by(desc(Livro.ano))

for linha in session.execute(query):
    print(linha.ano, linha.titulo)

1966 Flores para Algernon
1949 1984
1945 A Revolução dos Bichos
1881 Memórias Póstumas de Brás Cubas


`limit()` limita a quantidade de elementos retornados. Combinado com `order_by()`, podemos pegar os "2 livros mais antigos":

In [ ]:
query = select(Livro).order_by(Livro.ano).limit(2)
print(session.execute(query).all())

[(Livro(5, 'Memórias Póstumas de Brás Cubas', 'Machado de Assis', 1881),), (Livro(4, 'A Revolução dos Bichos', 'George Orwell', 1945),)]


**Agregações** <br>
O SQLAlchemy possui os módulos e métodos para auxiliar agregações e ordenações.
* [`func`](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.func): Dá acesso às funções SQL como `count`, `max`, `avg`, etc.
* [`asc`](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.asc) e [`desc`](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.desc): auxiliares do ORDER BY, respectivamente ascendente e descendente.

Com `group_by()` agrupamos os registros; `label()` dá um nome à coluna calculada. Quantos livros temos de cada autor?

In [ ]:
# SELECT autor, COUNT(id) AS qtd FROM Livro
query = (
    select(Livro.autor, func.count(Livro.id).label("qtd"))
    .group_by(Livro.autor)
    .order_by(desc("qtd"))
)

for row in session.execute(query):
    print(row.autor, '->', row.qtd)

George Orwell -> 2
Machado de Assis -> 1
Daniel Keyes -> 1


Sem `group_by`, a função agrega a tabela inteira. Para receber um único valor, usamos `session.scalar()`:

In [ ]:
total = session.scalar(select(func.count(Livro.id)))
mais_recente = session.scalar(select(func.max(Livro.ano)))

print("Total de livros:", total)
print("Ano do livro mais recente:", mais_recente)

Total de livros: 4
Ano do livro mais recente: 1966


### `close()`

Por fim, nunca esqueça de finalizar a sessão.

In [ ]:
session.close()

### Exercício

Vamos praticar com uma nova tabela de **alunos**.

**1. Modelagem.** Crie a classe `Aluno`, mapeada para a tabela `alunos`, com as colunas:
* `id`: inteiro, chave primária
* `nome`: texto
* `curso`: texto
* `nota`: número decimal (`Float`, importado de `sqlalchemy`)

**2. Criação.** Crie a tabela no banco com `Base.metadata.create_all(engine)`.

**3. Inserção.** Insira os 5 alunos abaixo com `add_all()` e faça o `commit()`:

| nome | curso | nota |
|---|---|---|
| Ana | Informática | 9.0 |
| Bruno | Informática | 6.5 |
| Carla | Eletrônica | 7.5 |
| Diego | Eletrônica | 5.0 |
| Elisa | Informática | 8.0 |

**4. Consultas.** Escreva e execute uma consulta para cada item:
* a) Todos os alunos
* b) Apenas o `nome` e a `nota` de cada aluno
* c) Alunos do curso "Informática"
* d) Alunos com nota maior ou igual a 7, da maior para a menor nota
* e) Quantidade de alunos por curso

In [10]:
# 1 e 2. Modelagem e criação da tabela
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy import select, update, delete, insert, func, asc, desc
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine('sqlite:///escola.db')
Base = declarative_base()

class Aluno(Base):
    __tablename__ = 'alunos'
    id = Column(Integer, primary_key=True)
    nome = Column(String)
    curso = Column(String)
    nota = Column(Float)

    def __repr__(self):
        return f"Aluno({self.id}, {self.nome!r}, {self.curso!r}, {self.nota})"

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)


In [11]:
# 3. Inserção dos alunos
Session = sessionmaker(bind=engine)
session = Session()

alunos = [
  Aluno(nome='Ana', curso='Informática', nota=9.0),
  Aluno(nome='Bruno', curso='Informática', nota=6.5),
  Aluno(nome='Carla', curso='Eletrônica', nota=7.5),
  Aluno(nome='Diego', curso='Eletrônica', nota=5.0),
  Aluno(nome='Elisa', curso='Informática', nota=8.0),
]

session.add_all(alunos)
session.commit()
print(f"{len(alunos)} alunos inseridos")


session.close()


5 alunos inseridos


In [12]:
# 4a. Todos os alunos
query = select(Aluno)
res = session.execute(query).all()

for linha in res:
    print(linha.Aluno)



Aluno(1, 'Ana', 'Informática', 9.0)
Aluno(2, 'Bruno', 'Informática', 6.5)
Aluno(3, 'Carla', 'Eletrônica', 7.5)
Aluno(4, 'Diego', 'Eletrônica', 5.0)
Aluno(5, 'Elisa', 'Informática', 8.0)


In [14]:
# 4b. Apenas nome e nota
query = select(Aluno.nome, Aluno.nota)
res = session.execute(query).all()

for linha in res:
  print(linha.nome, '-', linha.nota)

Ana - 9.0
Bruno - 6.5
Carla - 7.5
Diego - 5.0
Elisa - 8.0


In [19]:
# 4c. Alunos de Informática
query = select(Aluno).where(Aluno.curso == "Informática")
res = session.execute(query).all()

for linha in res:
  print(linha.Aluno)

Aluno(1, 'Ana', 'Informática', 9.0)
Aluno(2, 'Bruno', 'Informática', 6.5)
Aluno(5, 'Elisa', 'Informática', 8.0)


In [ ]:
# 4d. Nota >= 7, da maior para a menor


In [ ]:
# 4e. Quantidade de alunos por curso
